# Musical Chord Recognition with `puar-playground/btc-chord`

This notebook performs automatic chord recognition using the **BTC (Bi-directional Transformer for Chord Recognition)** model from Hugging Face (`puar-playground/btc-chord`).

### Pipeline:
1. **Load Separated Stems**: Reads `data/bass.wav` and `data/other.wav` produced by Demucs.
2. **Combine Stems**: Merges **bass + other** into `data/accompaniment.wav` for cleaner chord estimation without drum/vocal interference.
3. **Predict Chords on GPU with Visual Progress**: Runs chunk-by-chunk transformer inference with a live `tqdm` progress bar.
4. **Export Results**: Saves chord timeline directly to **`data/chords.csv`**.

> **Kernel**: Make sure the kernel is set to **`Python (seperate)`**.

## 1. Setup & Device Configuration

In [1]:
from pathlib import Path
import torch
import numpy as np
import soundfile as sf
import pandas as pd
from tqdm.auto import tqdm

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 5090


/home/cglab/miniconda3/envs/seperate/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Define Relative Paths

In [2]:
data_dir = Path("data")

# Input stems
bass_path = data_dir / "bass.wav"
other_path = data_dir / "other.wav"

# Combined accompaniment output
accompaniment_path = data_dir / "bass_other.wav"

# Chord recognition output file
chords_csv_path = data_dir / "chords.csv"

print(f"Bass path           : {bass_path}")
print(f"Other path          : {other_path}")
print(f"Accompaniment target: {accompaniment_path}")
print(f"Chords CSV target   : {chords_csv_path}")

assert bass_path.exists(), f"Missing stem: {bass_path}. Please run separate.ipynb first!"
assert other_path.exists(), f"Missing stem: {other_path}. Please run separate.ipynb first!"

Bass path           : data/bass.wav
Other path          : data/other.wav
Accompaniment target: data/bass_other.wav
Chords CSV target   : data/chords.csv


## 3. Combine `bass` and `other` into Harmonic Accompaniment
We load both audio stems, sum their signals, normalize if needed to avoid clipping, and save to `data/accompaniment.wav`.

In [3]:
# Load audio files
bass_audio, sr_bass = sf.read(bass_path)
other_audio, sr_other = sf.read(other_path)

assert sr_bass == sr_other, f"Sample rate mismatch: {sr_bass} vs {sr_other}"
sr = sr_bass

# Ensure equal length
min_len = min(len(bass_audio), len(other_audio))
combined = bass_audio[:min_len] + other_audio[:min_len]

# Avoid clipping by normalizing if peak exceeds 1.0
max_val = np.max(np.abs(combined))
if max_val > 1.0:
    combined = combined / max_val

# Save combined accompaniment
sf.write(accompaniment_path, combined, sr)
file_size_mb = accompaniment_path.stat().st_size / (1024 * 1024)
duration_sec = len(combined) / sr

print(f"✅ Successfully combined bass and other!")
print(f"   Output   : {accompaniment_path}")
print(f"   Size     : {file_size_mb:.2f} MB")
print(f"   Duration : {duration_sec:.2f} seconds ({duration_sec / 60:.2f} min)")

✅ Successfully combined bass and other!
   Output   : data/bass_other.wav
   Size     : 45.46 MB
   Duration : 270.24 seconds (4.50 min)


## 4. Load `puar-playground/btc-chord` on GPU

In [4]:
from transformers import AutoModel

print(f"Loading BTC Chord model from puar-playground/btc-chord on device='{device}'...")
model = AutoModel.from_pretrained(
    "puar-playground/btc-chord",
    trust_remote_code=True,
    device=device,
    large_voca=True  # 170-chord vocabulary
)
print("Model loaded successfully!")

Loading BTC Chord model from puar-playground/btc-chord on device='cuda'...


Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 4161.02it/s]

Model loaded successfully!


## 5. Run Chord Recognition with Live Visual Progress Bar

In [5]:
import importlib

@torch.no_grad()
def predict_with_progress(btc_wrapper, audio_file):
    """Predict chords with a live visual tqdm progress bar across audio chunks."""
    features = importlib.import_module("btc_src.features")
    
    print("1/2 Extracting log-CQT audio spectrogram features...")
    feat = features.audio_to_features(
        str(audio_file),
        sr_target=22050,
        inst_len=10.0,
        n_bins=144,
        bins_per_octave=24,
        hop_length=2048,
    )
    feat = feat.T
    feat = (feat - btc_wrapper._mean) / btc_wrapper._std
    
    timestep = 108
    time_unit = 10.0 / timestep
    n = timestep
    
    num_pad = n - (feat.shape[0] % n)
    feat = np.pad(feat, ((0, num_pad), (0, 0)), mode="constant", constant_values=0)
    num_instance = feat.shape[0] // n
    
    x = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(btc_wrapper._device)
    lines = []
    start_time = 0.0
    prev = None
    
    print(f"2/2 Running Transformer inference across {num_instance} audio chunks:")
    for t in tqdm(range(num_instance), desc="Predicting Chords", unit="chunk"):
        attn_out, _ = btc_wrapper.model.self_attn_layers(x[:, n * t : n * (t + 1), :])
        pred, _ = btc_wrapper.model.output_layer(attn_out)
        pred = pred.squeeze()
        
        for i in range(n):
            if t == 0 and i == 0:
                prev = pred[i].item()
                continue
            cur = pred[i].item()
            if cur != prev:
                end = time_unit * (n * t + i)
                lines.append({
                    "start": round(start_time, 3),
                    "end": round(end, 3),
                    "chord": btc_wrapper._idx_to_chord[prev]
                })
                start_time = end
                prev = cur
            if t == num_instance - 1 and i + num_pad == n:
                end = time_unit * (n * t + i)
                if start_time != end:
                    lines.append({
                        "start": round(start_time, 3),
                        "end": round(end, 3),
                        "chord": btc_wrapper._idx_to_chord[prev]
                    })
                break
    return lines

# Execute with progress bar
raw_chords = predict_with_progress(model, accompaniment_path)

# Convert to DataFrame
df_chords = pd.DataFrame(raw_chords)
df_chords["duration"] = (df_chords["end"] - df_chords["start"]).round(3)

print(f"\nExtracted {len(df_chords)} chord segments.\n")
print("First 15 chord segments:")
print(df_chords.head(15).to_string(index=False))

1/2 Extracting log-CQT audio spectrogram features...


/home/cglab/miniconda3/envs/seperate/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=327
  warnings.warn(
/home/cglab/miniconda3/envs/seperate/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=164
  warnings.warn(
/home/cglab/miniconda3/envs/seperate/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=82
  warnings.warn(


2/2 Running Transformer inference across 28 audio chunks:


Predicting Chords: 100%|██████████| 28/28 [00:00<00:00, 106.31chunk/s]


Extracted 253 chord segments.

First 15 chord segments:
 start    end  chord  duration
 0.000  4.630      N     4.630
 4.630  6.111 D#:min     1.481
 6.111  8.056      B     1.945
 8.056 10.000     G#     1.944
10.000 10.741     A#     0.741
10.741 11.296     G#     0.555
11.296 11.389     D#     0.093
11.389 12.870 D#:min     1.481
12.870 14.630      B     1.760
14.630 16.389     G#     1.759
16.389 18.333 A#:min     1.944
18.333 20.000     G#     1.667
20.000 21.759     D#     1.759
21.759 23.241     F#     1.482
23.241 25.093  F:min     1.852


## 6. Save Predictions to CSV
Save the structured chord timeline directly to `data/chords.csv`.

In [6]:
df_chords.to_csv(chords_csv_path, index=False)
print(f"✅ Successfully saved chords to: {chords_csv_path}")

✅ Successfully saved chords to: data/chords.csv


## 7. Chord Summary & Progression Overview

In [7]:
# Filter out 'N' (no-chord) to see most frequent chords
active_chords = df_chords[df_chords["chord"] != "N"]
chord_counts = active_chords.groupby("chord")["duration"].sum().sort_values(ascending=False)

print("Top Chords by Total Duration (seconds):")
for chord, dur in chord_counts.head(10).items():
    print(f"  • {chord:<10} : {dur:.2f} s")

# Simplified progression (consecutive unique chords)
progression = []
for chord in df_chords["chord"]:
    if not progression or progression[-1] != chord:
        progression.append(chord)

print(f"\nChord Progression Sequence ({len(progression)} transitions):")
print(" -> ".join(progression[:20]) + (" ..." if len(progression) > 20 else ""))

Top Chords by Total Duration (seconds):
  • F#         : 25.84 s
  • A#:min     : 22.41 s
  • B:min      : 22.13 s
  • G          : 21.76 s
  • D#         : 17.68 s
  • F:min      : 17.22 s
  • B          : 13.15 s
  • E          : 12.13 s
  • D#:min     : 11.66 s
  • A#         : 11.30 s

Chord Progression Sequence (253 transitions):
N -> D#:min -> B -> G# -> A# -> G# -> D# -> D#:min -> B -> G# -> A#:min -> G# -> D# -> F# -> F:min -> E -> D# -> G#:min -> D# -> D#:aug ...
